# N08 · Prefill 与 Decode：把“服务慢”拆成阶段


> 学习方式建议：先读“心智模型”，再运行代码实验，最后做练习题。每道题都附答案解析，不是为了考倒你，而是为了暴露最常见的模棱两可点。  
> 本 notebook 只做概念和可复现小实验；真实工程闭环请回到对应 lab 运行 `make smoke M=...` 并查看 `runs/.../metrics.jsonl`、日志和报告。


## 本节要解决的问题

LLM serving 的一次请求不是一个黑盒。它至少包含：排队、tokenize、prefill、首 token、decode、多 token streaming、结束处理。用户说“慢”，你要先问：是首 token 慢，还是后续 token 慢？

本节目标：区分 TTFT、ITL/TPOT、E2E latency，并知道它们分别对应哪些系统瓶颈。


## 学习地图与版本说明（截至 2026-04-30）

本节把一次 LLM 请求拆成阶段：入队、tokenize/chat template、prefill、首 token、decode 循环、streaming、结束处理。Prefill 通常处理整段 prompt，更像大矩阵计算；decode 每次只生成一个或少量 token，更容易受 KV cache 读写、调度、batch 组成和显存带宽影响。因此“慢”必须先翻译成 TTFT 高、ITL 高、E2E 高还是排队高。

版本上，本教程参考 vLLM stable metrics 文档和 SGLang PD disaggregation 文档。现代 serving 系统会通过 continuous batching、chunked prefill、prefix cache、prefill/decode 分离等机制在 TTFT、吞吐和 ITL 之间取舍。不同框架指标名会变化，但核心观测维度稳定：请求到达率、prompt/output token 分布、队列长度、cache 命中、首 token 延迟和逐 token 延迟。

学完本节，你应该能设计可信 benchmark：固定输入/输出 token 分布，区分 warmup 和采样窗口，报告并发/到达率，说明是否 streaming、是否启用 prefix cache、是否包含 tokenizer 时间。没有这些条件的 requests/sec 对比，通常不能支持工程结论。


## 1. 请求生命周期

```text
HTTP 到达
  → tokenizer / chat template
  → scheduler 排队
  → prefill：处理完整 prompt，生成首个可用 KV cache
  → first token：返回第一个 token
  → decode：每步生成一个或少量 token
  → stream / finish
```

指标对应关系：

- **TTFT**：从请求到达或开始处理，到第一个 token 返回。
- **ITL / TPOT**：decode 阶段相邻输出 token 的间隔。
- **E2E latency**：整个请求完成时间。
- **Queue time**：scheduler 等待时间，常被误算进 TTFT。


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

def latency_model(prompt_tokens, output_tokens, queue_ms=20, prefill_ms_per_token=0.08, decode_ms_per_token=7.0):
    prefill = prompt_tokens * prefill_ms_per_token
    ttft = queue_ms + prefill
    decode = output_tokens * decode_ms_per_token
    e2e = ttft + decode
    return {"prompt_tokens": prompt_tokens, "output_tokens": output_tokens, "TTFT_ms": ttft, "ITL_ms": decode_ms_per_token, "E2E_ms": e2e}

rows = [latency_model(p, o) for p in [128, 1024, 4096, 16384] for o in [32, 128, 512]]
df = pd.DataFrame(rows)
display(df.round(1).head(12))
df[df["output_tokens"] == 128].plot(x="prompt_tokens", y="TTFT_ms", marker="o", title="prompt length 对 TTFT 的影响")
plt.ylabel("ms")
plt.show()


## 2. Prefill 为什么更像“算力密集”？

Prefill 要一次性处理整个 prompt。对于长 prompt，矩阵乘法规模大，GPU 算力利用往往较高。它决定了首 token 前需要多久把上下文“读完”。

优化方向包括：

- prefix cache 命中，避免重复 prefill。
- chunked prefill，避免长 prompt 独占调度。
- 更好的 batching，把多个 prefill 合并。
- 减少无效 prompt token，例如过长系统提示、重复上下文。


## 3. Decode 为什么更像“带宽/调度密集”？

Decode 每次通常只新增少量 token，但要读取历史 KV cache。长上下文、高并发时，decode 受 KV cache 读取、memory bandwidth、scheduler、batch 组成影响很大。

优化方向包括：

- 控制并发和 max_new_tokens。
- KV cache 量化/分页/淘汰策略。
- continuous batching。
- speculative decoding（适用时）。
- PD disaggregation 中给 decode 合适 GPU 池。


In [ ]:
# 模拟：长 prompt 影响 TTFT，长 output 影响 E2E 和 decode 总时间。
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for out in [32, 128, 512]:
    part = df[df["output_tokens"] == out]
    axes[0].plot(part["prompt_tokens"], part["TTFT_ms"], marker="o", label=f"out={out}")
axes[0].set_title("TTFT 主要随 prompt 增长")
axes[0].set_xscale("log", base=2)
axes[0].legend()

for prompt in [128, 4096, 16384]:
    part = df[df["prompt_tokens"] == prompt]
    axes[1].plot(part["output_tokens"], part["E2E_ms"], marker="o", label=f"prompt={prompt}")
axes[1].set_title("E2E 同时受输出长度影响")
axes[1].legend()
plt.tight_layout()
plt.show()


## 4. benchmark 设计：必须固定输入/输出长度

很多 serving benchmark 的结论不可信，是因为没有固定或报告：

- prompt token 分布。
- output token 分布。
- 并发/请求到达率。
- 是否 streaming。
- 是否命中 prefix cache。
- 是否包含 tokenizer/chat template 时间。
- warmup 和采样窗口。

如果一个版本 TTFT 变差，要先问 prompt 是否更长、queue 是否更高、prefix cache 是否 miss，而不是直接怪模型。


## 5. 与本课程的连接

- L07 vLLM：建立 TTFT/ITL baseline。
- L08 SGLang：repeated-prefix benchmark 观察 cache 对 TTFT 的影响。
- L09 SGLang PD：比较 unified vs prefill/decode 分离。
- Debug ticket：`sglang_high_ttft_001`、`sglang_pd_decode_starve_004`。


## 6. 企业面试/工程判断痛点题（带答案）

### 题 1：TTFT 高、ITL 正常，说明什么？

**答案解析：** 更可能是 queue/prefill/tokenization/prefix cache 问题，而不是 decode 逐 token 生成问题。

### 题 2：ITL 高、TTFT 正常，说明什么？

**答案解析：** 更可能是 decode 阶段瓶颈：KV cache 读取、decode batch 组成、并发太高、memory bandwidth 或调度问题。

### 题 3：为什么只看 requests/sec 不够？

**答案解析：** 高 requests/sec 可能来自短 prompt/短 output；用户体验还取决于 TTFT、ITL、E2E、tail latency。必须按 workload 分层看。

### 题 4：长 prompt 请求会伤害短请求吗？

**答案解析：** 会。若 scheduler 没有 chunked prefill 或优先级策略，长 prefill 可能占用 GPU，导致短请求 TTFT 变差。

### 题 5：PD disaggregation 一定提升性能吗？

**答案解析：** 不一定。只有 workload 中 prefill/decode 资源画像明显不同，且路由/传输/资源比例合理时才收益。否则增加复杂度和传输开销。


## 参考资料

- vLLM metrics design: https://docs.vllm.ai/en/stable/design/metrics/
- SGLang PD disaggregation: https://docs.sglang.io/docs/advanced_features/pd_disaggregation
- NVIDIA Dynamo SGLang disaggregation: https://docs.nvidia.com/dynamo/dev/backends/sglang/sglang-disaggregation.html
